In [1]:
# -*- coding=utf-8 -*-
# __author = 'chentingting'__

import datetime as dt
import os, sys
sys.path.append('/Users/abakus/PycharmProjects/bacc_datalab')
import pandas as pd
import numpy as np
import plotly as ply

ply.offline.init_notebook_mode(connected=True)

from dateutil.relativedelta import relativedelta
from datalab_lib.common import common_func, data_util, const, common_plot, common_htmlTable
import re
from openpyxl.styles import numbers
from string import ascii_uppercase
import calendar

import warnings
warnings.filterwarnings('ignore')
#把不必要的提示錯誤關掉
import colorlover as cl
import copy
from plotly.offline import init_notebook_mode
import plotly.offline as py
init_notebook_mode(connected=True)
import plotly.graph_objects as go
import plotly.figure_factory as ff
from IPython.display import Markdown, display, HTML
from dateutil.parser import parse



today = dt.datetime.today()
current_month = today.strftime('%Y-%m')
start_date = (today.date() - relativedelta(days=7)).strftime('%Y-%m-%d')

day = calendar.monthrange(today.year,today.month)[1]
end_date = dt.datetime(today.year ,today.month,day).strftime('%Y-%m-%d')

# overview

In [2]:
@data_util.db_source(db='novadaxbi')
def data_overview_new1(update=False):
    """overview transaction 部分"""
    sql = """    
    select 
        'week' as type
        , subdate(date(order_end_time),if(date_format(date(order_end_time),'%%w')=0,7,date_format(date(order_end_time),'%%w'))-1) as period
        , worth_coin
        , count(DISTINCT order_id) as "tx_case_cnt"
        , count(DISTINCT cid) as "tx_customer_cnt"
        , sum(transaction_volume) as "tx_amount"
        , sum(transaction_volume)/count(DISTINCT order_id) as "avg_case_amout"
        , count(DISTINCT order_id)/count(DISTINCT cid) as "avg_customer_case"
    from bacc_nova.ba_dwd_success_transaction  
    where order_end_time>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day 
        and order_end_time<current_date and worth_coin='BRL'
    group by 1,2,3
    union 
    select 
        'month' as type
        , date_format(last_day(order_end_time),'%%Y-%%m-%%d') as period
        , worth_coin
        , count(DISTINCT order_id) as "tx_case_cnt"
        , count(DISTINCT cid) as "tx_customer_cnt"
        , sum(transaction_volume) as "tx_amount"
        , sum(transaction_volume)/count(DISTINCT order_id) as "avg_case_amout"
      , count(DISTINCT order_id)/count(DISTINCT cid) as "avg_customer_case"
    from bacc_nova.ba_dwd_success_transaction  
    where order_end_time>=date_format(current_date,'%%Y-%%m-01') - interval '7' month 
    and order_end_time<current_date
    and worth_coin='BRL'
    group by 1,2,3
    order by 1,2 desc,3
    """
    return sql



@data_util.db_source(db='novadaxbi')
def data_overview_new2(update=False):
    """overview asset 部分"""
    sql = """    
    select 
        'week' as type
        , subdate(date(order_end_time),if(date_format(date(order_end_time),'%%w')=0,7,date_format(date(order_end_time),'%%w'))-1) as period
        , is_otc
        , worth_coin
        , record_type
        , sum(transaction_volume) as tx_amount_brl
        , sum(transaction_volume / price) as tx_amount_btc
    from bacc_nova.ba_dwd_success_transaction m1 
    join bacc_nova.dim_exchange_rate m2 on m2.settle_date=date(m1.order_end_time) and digital_code='BTC'
    where order_end_time>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day and order_end_time<current_date
    and order_type in ('Fiat_DW','COIN_IN_OUT') and worth_coin=m2.legal_code
    and m1.worth_coin='BRL'
    group by 1,2,3,4,5
    union 
    select 
        'month' as type
        , date_format(last_day(order_end_time),'%%Y-%%m-%%d') as period
        , is_otc
        , worth_coin
        , record_type
        , sum(transaction_volume) as tx_amount_brl
        , sum(transaction_volume / price) as tx_amount_btc
    from bacc_nova.ba_dwd_success_transaction m1
    join bacc_nova.dim_exchange_rate m2 on m2.settle_date=date(m1.order_end_time) and digital_code='BTC'
    where order_end_time>=date_format(current_date,'%%Y-%%m-01') - interval '7' month and order_end_time<current_date
    and order_type in ('Fiat_DW','COIN_IN_OUT') and worth_coin=m2.legal_code
    and m1.worth_coin='BRL'
    group by 1,2,3,4,5
    order by 1,4,3,5,2 desc 
    """
    return sql



@data_util.db_source(db='novadaxbi')
def data_overview_new3(update=False):
    """overview 获客 转化漏斗部分"""
    sql = """    
    select 
        'week' as type
        , subdate(date(m1.register_time),if(date_format(date(m1.register_time),'%%w')=0,7,date_format(date(m1.register_time),'%%w'))-1) as period
        , market_source
        , register_device_type
        , m1.kyc_area
        , m1.kyc_type
        , count(DISTINCT m1.cid) as register_counts
        , count(DISTINCT if(subdate(date(min_kyc1_submit_time),if(date_format(date(min_kyc1_submit_time),'%%w')=0,7,date_format(date(min_kyc1_submit_time),'%%w'))-1) = subdate(date(register_time),if(date_format(date(register_time),'%%w')=0,7,date_format(date(register_time),'%%w'))-1),m1.cid,null)) as period_try_kyc1_counts
        , count(DISTINCT if(subdate(date(min_kyc1_success_time),if(date_format(date(min_kyc1_success_time),'%%w')=0,7,date_format(date(min_kyc1_success_time),'%%w'))-1) = subdate(date(register_time),if(date_format(date(register_time),'%%w')=0,7,date_format(date(register_time),'%%w'))-1),m1.cid,null)) as period_kyc1_counts 
        , count(DISTINCT if(subdate(date(min_try_deposit_time),if(date_format(date(min_try_deposit_time),'%%w')=0,7,date_format(date(min_try_deposit_time),'%%w'))-1) = subdate(date(register_time),if(date_format(date(register_time),'%%w')=0,7,date_format(date(register_time),'%%w'))-1),m1.cid,null)) as period_try_deposit_counts
        , count(DISTINCT if(subdate(date(min_trade_all_time),if(date_format(date(min_trade_all_time),'%%w')=0,7,date_format(date(min_trade_all_time),'%%w'))-1) = subdate(date(register_time),if(date_format(date(register_time),'%%w')=0,7,date_format(date(register_time),'%%w'))-1),m1.cid,null)) as period_tot_counts
        , count(DISTINCT if(subdate(date(min_deposit_time),if(date_format(date(min_deposit_time),'%%w')=0,7,date_format(date(min_deposit_time),'%%w'))-1) = subdate(date(register_time),if(date_format(date(register_time),'%%w')=0,7,date_format(date(register_time),'%%w'))-1),m1.cid,null)) as period_deposit_counts
        , count(DISTINCT if(subdate(date(min_trade_time),if(date_format(date(min_trade_time),'%%w')=0,7,date_format(date(min_trade_time),'%%w'))-1) = subdate(date(register_time),if(date_format(date(register_time),'%%w')=0,7,date_format(date(register_time),'%%w'))-1),m1.cid,null)) as period_trade_counts 
    from bacc_nova.ba_dwd_customer m1 
    where m1.register_time>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day 
    and m1.register_time<current_date
    and m1.kyc_area !='EUROPE'
    group by 1,2,3,4,5,6
    union 
    select 
        'month' as type
        , date_format(m1.register_time,'%%Y-%%m') as period
        , market_source
        , register_device_type
        , m1.kyc_area
        , m1.kyc_type
        , count(DISTINCT m1.cid) as register_counts
        , count(DISTINCT if(date_format(m1.min_kyc1_submit_time,'%%Y-%%m')=date_format(m1.register_time,'%%Y-%%m'),m1.cid,null)) as period_try_kyc1_counts
        , count(DISTINCT if(date_format(m1.min_kyc1_success_time,'%%Y-%%m')=date_format(m1.register_time,'%%Y-%%m'),m1.cid,null)) as period_kyc1_counts
        , count(DISTINCT if(date_format(m1.min_try_deposit_time,'%%Y-%%m')=date_format(m1.register_time,'%%Y-%%m'),m1.cid,null)) as period_try_deposit_counts
        , count(DISTINCT if(date_format(m1.min_trade_all_time,'%%Y-%%m')=date_format(m1.register_time,'%%Y-%%m'),m1.cid,null)) as period_tot_counts
        , count(DISTINCT if(date_format(m1.min_deposit_time,'%%Y-%%m')=date_format(m1.register_time,'%%Y-%%m'),m1.cid,null)) as period_deposit_counts
        , count(DISTINCT if(date_format(m1.min_trade_time,'%%Y-%%m')=date_format(m1.register_time,'%%Y-%%m'),m1.cid,null)) as period_trade_counts
    from bacc_nova.ba_dwd_customer m1 
    where m1.register_time>=date_format(current_date,'%%Y-%%m-01') - interval '7' month 
    and m1.register_time<current_date
    and m1.kyc_area !='EUROPE'
    group by 1,2,3,4,5,6
    order by 1,2 desc,3
    """
    return sql



@data_util.db_source(db='novadaxbi')
def data_overview_new4(update=False):
    """overview 获客 首次交易部分"""
    sql = """    
    select 
        'week' as type
        , subdate(date(m1.order_end_time),if(date_format(date(m1.order_end_time),'%%w')=0,7,date_format(date(m1.order_end_time),'%%w'))-1) as period
        , worth_coin
        , market_source
        , count(DISTINCT m1.cid) as new_user_cnt
    from bacc_nova.ba_dwp_first_trade_all m1 
    left join bacc_nova.ba_dwd_customer m2 on m2.cid=m1.cid 
    where m1.order_end_time>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day 
    and m1.order_end_time<current_date
    and m2.kyc_area !='EUROPE'
    group by 1,2,3,4
    union 
    select 
        'month' as type
        , date_format(last_day(m1.order_end_time),'%%Y-%%m-%%d') as period
        , worth_coin 
        , market_source
        , count(DISTINCT m1.cid) as new_user_cnt
    from bacc_nova.ba_dwp_first_trade_all m1 
    left join bacc_nova.ba_dwd_customer m2 on m2.cid=m1.cid 
    where m1.order_end_time>=date_format(current_date,'%%Y-%%m-01') - interval '7' month 
    and m1.order_end_time<current_date
    and m2.kyc_area !='EUROPE'
    group by 1,2,3,4
    order by 1,3,2 desc,4
    """
    return sql


def calc(df, index, sort_column=None):
    df = df.drop(['type'],axis=1)
    df = df.pivot_table(index = index,columns = 'period', fill_value=0).stack(0).reset_index()
    if sort_column == None:
        df = df.sort_index(axis=1,ascending=False)
    else:
        df = df.sort_values(sort_column,ascending=False).sort_index(axis=1,ascending=False)
    
    return df

In [3]:
## transaction
df_transaction = data_overview_new1()
df_transaction_week = df_transaction[df_transaction['type']=='week'].append(df_transaction.head(1))
df_transaction_week = calc(df_transaction_week,index = 'worth_coin',sort_column = 'level_1')

df_transaction_month = df_transaction[(df_transaction['type']=='month') & (df_transaction['period']!=end_date)]
df_transaction_month = calc(df_transaction_month,index = 'worth_coin',sort_column = 'level_1')



## asscet
df_asset = data_overview_new2()
df_asset = df_asset.replace({'DEPOSIT':'1_DEPOSIT','COIN_IN':'2_COIN_IN',
                             'WITHDRAW':'3_WITHDRAW','COIN_OUT':'4_COIN_OUT'})

df_asset_week = df_asset[(df_asset['type'] == 'week')].append(df_asset[df_asset['period'] == end_date])
df_asset_week = calc(df_asset_week,index = ['worth_coin','is_otc','record_type'],sort_column = 'level_3')

df_asset_month = df_asset[(df_asset['type'] == 'month') & (df_asset['period'] != end_date)]
df_asset_month = calc(df_asset_month,index = ['worth_coin','is_otc','record_type'],sort_column = 'level_3')



## customer acquisition
### 转化漏斗
df_conv_detail = data_overview_new3()
### 首次买卖存取
df_first_trade = data_overview_new4()
df_first_trade_month = df_first_trade[(df_first_trade['type']=='month')&(df_first_trade['period'] != end_date)]
df_first_trade_month = calc(df_first_trade_month,index = ['worth_coin','market_source']).drop('level_2',axis=1)

df_first_trade_week = df_first_trade[(df_first_trade['type']=='week')|(df_first_trade['period'] == end_date)]
df_first_trade_week = calc(df_first_trade_week,index = ['worth_coin','market_source']).drop('level_2',axis=1)

In [4]:
df_conv_detail[(df_conv_detail['period'] == '2021-08-16') & (df_conv_detail['market_source'] == 'Google_app')]

,type,period,market_source,register_device_type,kyc_area,kyc_type,register_counts,period_try_kyc1_counts,period_kyc1_counts,period_try_deposit_counts,period_tot_counts,period_deposit_counts,period_trade_counts
334,week,2021-08-16,Google_app,adr,BRAZIL,INDIVIDUAL,1215,915,862,238,241,238,184
335,week,2021-08-16,Google_app,adr,BRAZIL,INSTITUTION,9,0,0,0,0,0,0
336,week,2021-08-16,Google_app,adr,OTHER,INDIVIDUAL,6,0,0,0,0,0,0
337,week,2021-08-16,Google_app,adr,OTHER,INSTITUTION,3,0,0,0,0,0,0


# transaction

In [5]:
@data_util.db_source(db='novadaxbi')
def data_trans_type_new1(update=False):
    """按交易类型区分"""
    sql = """    
    select type,period,worth_coin,ifnull(trans_type,'total')trans_type,tx_case_cnt,tx_customer_cnt,tx_amount
    from 
    (
        select 
            'week' as type
            , subdate(date(order_end_time),if(date_format(date(order_end_time),'%%w')=0,7,date_format(date(order_end_time),'%%w'))-1) as period
            , worth_coin
            , case 
                when is_otc ='OTC' then 'OTC'
                when is_otc != 'OTC' and sub_order_type='api' then concat(order_type,'_',sub_order_type)
                when is_otc != 'OTC' and (sub_order_type !='api' or sub_order_type is null) then order_type 
            end as trans_type
            , count(DISTINCT order_id) as tx_case_cnt
            , count(DISTINCT cid) as tx_customer_cnt
            , sum(transaction_volume) as tx_amount
        from bacc_nova.ba_dwd_success_transaction
        where order_end_time>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day 
        and order_end_time<current_date and record_type in ('SELL','BUY')
        and worth_coin='BRL'
        group by 1,2,3,4 with rollup
        union 
        select 
            'month' as type
            , date_format(order_end_time,'%%Y-%%m') as period
            , worth_coin
            , case 
                when is_otc ='OTC' then 'OTC'
                when is_otc != 'OTC' and sub_order_type='api' then concat(order_type,'_',sub_order_type)
                when is_otc != 'OTC' and (sub_order_type !='api' or sub_order_type is null) then order_type 
            end as trans_type
            , count(DISTINCT order_id) as tx_case_cnt
            , count(DISTINCT cid) as tx_customer_cnt
            , sum(transaction_volume) as tx_amount
        from bacc_nova.ba_dwd_success_transaction
        where order_end_time>=date_format(curdate(),'%%Y-%%m-01') - interval '7' month 
        and order_end_time<current_date and record_type in ('SELL','BUY')
        and worth_coin='BRL'
        group by 1,2,3,4 with rollup 
    )a 
    where period is not null and worth_coin is not null
    order by 1,3,2 desc,4
    """
    return sql


@data_util.db_source(db='novadaxbi')
def data_trans_type_new2(update=False):
    """按交易币对区分"""
    sql = """    
    select 
        'week' as type
        , subdate(date(order_end_time),if(date_format(date(order_end_time),'%%w')=0,7,date_format(date(order_end_time),'%%w'))-1) as period
        , order_type
        , if(symbol like '%%BTC%%','BTC','NON_BTC') as symbol
        , symbol as subsymbol
        , count(DISTINCT order_id) as case_cnt
        , sum(transaction_volume) as amount
    from bacc_nova.ba_dwd_success_transaction
    where record_type in ('SELL','BUY') and is_otc != 'OTC' 
    and order_end_time>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '28' day 
    and order_end_time<current_date 
    and worth_coin='BRL'
    group by 1,2,3,4,5
    union
    select
        'month' as type
        , date_format(order_end_time,'%%Y-%%m-01') as period
        , order_type
        , if(symbol like '%%BTC%%','BTC','NON_BTC') as symbol
        , symbol as subsymbol
        , count(DISTINCT order_id) as case_cnt
        , sum(transaction_volume) as amount
    from bacc_nova.ba_dwd_success_transaction
    where record_type in ('SELL','BUY') and is_otc != 'OTC' 
    and order_end_time>=date_format(curdate(),'%%Y-%%m-01') - interval '4' month 
    and order_end_time<date_format(curdate(),'%%Y-%%m-01')
    and worth_coin='BRL'
    group by 1,2,3,4,5
    order by 1,2
    """
    return sql


@data_util.db_source(db='novadaxbi')
def data_trans_type_new3(update=False):
    """按交易设备区分"""
    sql = """    
    select 
        'week' as type
        , subdate(date(order_end_time),if(date_format(date(order_end_time),'%%w')=0,7,date_format(date(order_end_time),'%%w'))-1) as period
        , worth_coin
        , order_device_type
        , count(DISTINCT if(record_type='DEPOSIT',order_id,null)) as "存款笔数"
        , count(DISTINCT if(record_type='WITHDRAW',order_id,null)) as "取款笔数"
        , count(DISTINCT if(record_type='COIN_IN',order_id,null)) as "存币笔数"
        , count(DISTINCT if(record_type='COIN_OUT',order_id,null)) as "取币笔数"
        , count(DISTINCT if(record_type in ('SELL','BUY'),order_id,null)) as "买卖笔数"
        , count(DISTINCT if(order_type='Agent_Fiat_TX',order_id,null)) as "法币Agent笔数"
        , count(DISTINCT if(order_type='Orderbook_Fiat_TX',order_id,null)) as "法币orderbook笔数"
        , count(DISTINCT if(order_type='Crypto_TX',order_id,null)) as "币币orderbook笔数"
    from bacc_nova.ba_dwd_success_transaction
    where is_otc != 'OTC' and worth_coin='BRL'
    and order_end_time>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day and order_end_time<current_date 
    group by 1,2,3,4
    union 
    select 
        'month' as type
        , date_format(order_end_time,'%%Y-%%m') as period
        , worth_coin
        , order_device_type
        , count(DISTINCT if(record_type='DEPOSIT',order_id,null)) as "存款笔数"
        , count(DISTINCT if(record_type='WITHDRAW',order_id,null)) as "取款笔数"
        , count(DISTINCT if(record_type='COIN_IN',order_id,null)) as "存币笔数"
        , count(DISTINCT if(record_type='COIN_OUT',order_id,null)) as "取币笔数"
        , count(DISTINCT if(record_type in ('SELL','BUY'),order_id,null)) as "买卖笔数"
        , count(DISTINCT if(order_type='Agent_Fiat_TX',order_id,null)) as "法币Agent笔数"
        , count(DISTINCT if(order_type='Orderbook_Fiat_TX',order_id,null)) as "法币orderbook笔数"
        , count(DISTINCT if(order_type='Crypto_TX',order_id,null)) as "币币orderbook笔数"
    from bacc_nova.ba_dwd_success_transaction
    where is_otc != 'OTC' and worth_coin='BRL'
    and order_end_time>=date_format(current_date,'%%Y-%%m-01') - interval '7' month 
    and order_end_time<current_date 
    group by 1,2,3,4
    order by 1,2
    """
    return sql



@data_util.db_source(db='novadaxbi')
def data_trans_type_new4(update=False):
    """按OTC与否区分"""
    sql = """    
    select 
        'week' as type
        , subdate(date(order_end_time),if(date_format(date(order_end_time),'%%w')=0,7,date_format(date(order_end_time),'%%w'))-1) as period
        , is_otc
        , worth_coin
        , count(DISTINCT cid) as tx_customer_cnt
        , sum(transaction_volume) as tx_amount
    from bacc_nova.ba_dwd_success_transaction
    where order_end_time>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day 
    and order_end_time<current_date
    and worth_coin='BRL'
    group by 1,2,3,4
    union 
    select 
        'month' as type
        , date_format(order_end_time,'%%Y-%%m') as period
        , is_otc
        , worth_coin
        , count(DISTINCT cid) as tx_customer_cnt
        , sum(transaction_volume) as tx_amount
    from bacc_nova.ba_dwd_success_transaction
    where order_end_time>=date_format(curdate(),'%%Y-%%m-01') - interval '7' month 
    and order_end_time<current_date
    and worth_coin='BRL'
    group by 1,2,3,4
    """
    return sql

# 按交易类型
df_trans_type = data_trans_type_new1()

## 按买卖币对
df_symbol = data_trans_type_new2()

## 按设备
df_order_device = data_trans_type_new3()

## 区分OTC与否
df_order_otc = data_trans_type_new4()

df_order_otc_month = df_order_otc[(df_order_otc['type'] == 'month') & (df_order_otc['period'] != current_month)]

df_order_otc_month = df_order_otc_month.pivot_table(index=['worth_coin','is_otc'],columns='period',
                                                    values=['tx_customer_cnt','tx_amount'],
                                                    aggfunc='sum').stack(0)

df_order_otc_month = df_order_otc_month.drop(('BRL','OTC', 'tx_customer_cnt'),axis=0).reset_index()

df_order_otc_month = df_order_otc_month.sort_index(by='level_2',ascending=False).sort_index(axis=1,ascending=False)


df_order_otc_week = df_order_otc[(df_order_otc['type'] == 'week') | (df_order_otc['period'] == current_month)]
df_order_otc_week = df_order_otc_week.pivot_table(index=['worth_coin','is_otc'],columns='period',
                                                  values=['tx_customer_cnt','tx_amount'],
                                                  aggfunc='sum').stack(0)
# df_order_otc_week = df_order_otc_week.drop(('BRL','OTC', 'tx_customer_cnt'),axis=0).reset_index()

# df_order_otc_week = df_order_otc_week.sort_index(by='level_2',ascending=False).sort_index(axis=1,ascending=False)

df_order_otc_week = df_order_otc_week.reset_index().sort_index(axis=1,ascending=False)

# kyc

In [6]:
@data_util.db_source(db='novadaxbi')
def data_kyc1(update=False):
    """kyc审核成功率 & 时间"""
    sql = """    
    select 
        'week' as type,
        subdate(date(m1.created_at),if(date_format(date(m1.created_at),'%%w')=0,7,date_format(date(m1.created_at),'%%w'))-1) as period,
        m1.kyc_level,
        m1.kyc_area,
        m1.kyc_type,
        m1.auditor,
        count(DISTINCT m1.id) as total,
        count(DISTINCT m1.customer_id) as user,
        count(DISTINCT if(status = 'SUCCESSFUL',m1.id,null)) as suc,
        count(DISTINCT if(status = 'SUCCESSFUL',m1.customer_id,null)) as suc_cnt,
        count(DISTINCT m3.cid) as trade_all,
        count(DISTINCT m3.trade_cid) as trade,
        count(DISTINCT if(status = 'FAILED',m1.id,null)) as fail,
        count(DISTINCT if(status = 'FAILED',m1.customer_id,null)) as fail_cnt,
        count(DISTINCT if(status in ('PENDING', 'AUDITING'),m1.id,null)) as pending,
        sum(if(status in ('SUCCESSFUL','FAILED'),TIMESTAMPDIFF(minute,m1.created_at,m1.audited_at),0)) as total_deal_min
    from novadax.nova_kyc_audit_record m1
    left join novadax.nova_customer_ext m2 on m2.customer_id = m1.customer_id
    left join (select cid,if(record_type in ('SELL','BUY'),cid,null) as trade_cid from bacc_nova.ba_dwd_success_transaction where is_otc !='OTC' group by 1)m3 on m3.cid=m1.customer_id
    where m1.created_at>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '42' day and m1.created_at<current_date
    and m2.source <> 'TEST' and m1.kyc_area !='EUROPE'
    and m1.status !='CANCELED'
    group by 1,2,3,4,5,6
    union
    select 
        'month' as type,
        date_format(m1.created_at,'%%Y-%%m') as period,
        m1.kyc_level,
        m1.kyc_area,
        m1.kyc_type,
        m1.auditor,
        count(DISTINCT m1.id) as total,
        count(DISTINCT m1.customer_id) as user,
        count(DISTINCT if(status = 'SUCCESSFUL',m1.id,null)) as suc,
        count(DISTINCT if(status = 'SUCCESSFUL',m1.customer_id,null)) as suc_cnt,
        count(DISTINCT m3.cid) as trade_all,
        count(DISTINCT m3.trade_cid) as trade,
        count(DISTINCT if(status = 'FAILED',m1.id,null)) as fail,
        count(DISTINCT if(status = 'FAILED',m1.customer_id,null)) as fail_cnt,
        count(DISTINCT if(status in ('PENDING', 'AUDITING'),m1.id,null)) as pending,
        sum(if(status in ('SUCCESSFUL','FAILED'),TIMESTAMPDIFF(minute,m1.created_at,m1.audited_at),0)) as total_deal_min
    from novadax.nova_kyc_audit_record m1
    left join novadax.nova_customer_ext m2 on m2.customer_id = m1.customer_id
    left join (select cid,if(record_type in ('SELL','BUY'),cid,null) as trade_cid from bacc_nova.ba_dwd_success_transaction where is_otc !='OTC' group by 1)m3 on m3.cid=m1.customer_id
    where m2.source <> 'TEST' and m1.kyc_area !='EUROPE'
    and m1.status !='CANCELED'
    and m1.created_at>=date_format(curdate(),'%%Y-%%m-01') - interval '7' month 
    and m1.created_at<date_format(curdate(),'%%Y-%%m-01') 
    group by 1,2,3,4,5,6
    order by 5,4,3,6,2,1
    """
    return sql


@data_util.db_source(db='novadaxbi')
def data_kyc2(update=False):
    """kyc分设备"""
    sql = """    
    select 
        'week' as type,
        subdate(date(m1.created_at),if(date_format(date(m1.created_at),'%%w')=0,7,date_format(date(m1.created_at),'%%w'))-1) as period,
        if(m1.source_mark='undefined','web',m1.source_mark) as source_mark,
        count(DISTINCT m1.id) as total,
        count(DISTINCT if(status = 'SUCCESSFUL',m1.id,null)) as suc
    from novadax.nova_kyc_audit_record m1
    left join novadax.nova_customer_ext m2 on m2.customer_id = m1.customer_id
    where m1.created_at>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day and m1.created_at<current_date
    and m2.source <> 'TEST' and m1.kyc_level=1 and kyc_area='BRAZIL' and kyc_type='INDIVIDUAL'
    group by 1,2,3
    union 
    select 
        'month' as type,
        date_format(m1.created_at,'%%Y-%%m') as period,
        if(m1.source_mark='undefined','web',m1.source_mark) as source_mark,
        count(DISTINCT m1.id) as total,
        count(DISTINCT if(status = 'SUCCESSFUL',m1.id,null)) as suc
    from novadax.nova_kyc_audit_record m1
    left join novadax.nova_customer_ext m2 on m2.customer_id = m1.customer_id
    where m1.created_at>=date_format(current_date,'%%Y-%%m-01') - interval '7' month and m1.created_at<current_date
    and m2.source <> 'TEST' and m1.kyc_level=1 and kyc_area='BRAZIL' and kyc_type='INDIVIDUAL'
    group by 1,2,3
    order by 3,2,1

    """
    return sql


@data_util.db_source(db='novadaxbi')
def data_kyc3(update=False):
    """kyc分市场"""
    sql = """    
    select 
        'week' as type,
        subdate(date(m1.created_at),if(date_format(date(m1.created_at),'%%w')=0,7,date_format(date(m1.created_at),'%%w'))-1) as period,
        market_source,
        count(DISTINCT m1.id) as total,
        count(DISTINCT if(status = 'SUCCESSFUL',m1.id,null)) as suc
    from novadax.nova_kyc_audit_record m1
    left join bacc_nova.ba_dwd_customer m2 on m2.cid = m1.customer_id
    where m1.created_at>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day and m1.created_at<current_date
    and m1.kyc_level=1 and m1.kyc_area='BRAZIL' and m1.kyc_type='INDIVIDUAL'
    group by 1,2,3
    union 
    select 
        'month' as type,
        date_format(m1.created_at,'%%Y-%%m') as period,
        market_source,
        count(DISTINCT m1.id) as total,
        count(DISTINCT if(status = 'SUCCESSFUL',m1.id,null)) as suc
    from novadax.nova_kyc_audit_record m1
    left join bacc_nova.ba_dwd_customer m2 on m2.cid = m1.customer_id
    where m1.created_at>=date_format(current_date,'%%Y-%%m-01') - interval '7' month and m1.created_at<current_date
    and m1.kyc_level=1 and m1.kyc_area='BRAZIL' and m1.kyc_type='INDIVIDUAL'
    group by 1,2,3
    order by 3,2,1

    """
    return sql


# kyc 审核成功率 & 时长
df_kyc = data_kyc1()

# kyc 审核成功率 & 时长
df_kyc_device = data_kyc2()

# kyc 审核成功率 & 时长
df_kyc_market = data_kyc3()


# deposit & withdraw rate

In [7]:
@data_util.db_source(db='novadaxbi')
def data_deposit1(update=False):
    """首存首取成功率"""
    sql = """    
    select 
        'week' as type
        , subdate(date(created_at),if(date_format(date(created_at),'%%w')=0,7,date_format(date(created_at),'%%w'))-1) as period
        , record_type
        , asset_code
        , if(rnk=1,'first','not_first') as "rnk"
        , bank_name
        , bank_code
        , count(DISTINCT id) as "case"
        , count(DISTINCT if(record_status='SUCCESS',id,null)) as "succ"
        , count(DISTINCT if(record_status='FAIL',id,null)) as "fail"
        , count(DISTINCT if(record_status in ('NEDD_ADUIT','PAYED'),id,null)) as "pending"
        , sum(if(record_status in ('SUCCESS','FAIL'),TIMESTAMPDIFF(hour,created_at,finished_at),0)) as deal_hour
        , max(amount) max_amt
        , sum(amount) as total_amt
        , max(if(record_status='SUCCESS',amount,null)) succ_max_amt
        , sum(if(record_status='SUCCESS',amount,0)) as succ_amt
     from
    (
        select a.*,if(@mainid = customer_id and @maintype = record_type,@rn := @rn + 1,@rn :=1) as rnk,
            @mainid := customer_id,@maintype := record_type
        from 
        ( -- 用户首次存款取款记录
            select DISTINCT m1.id,m1.customer_id,
            if(m1.finished_at<m1.created_at,m1.finished_at,m1.created_at) created_at,
            m1.finished_at,m1.record_type,m1.record_status,m1.bank_name,
            concat(
            case 
              when m1.record_note like '%%ewally%%' then 'ewally' 
              when m1.record_note in ('transferAsset-out','transferAsset-in') then 'internal_transfer' 
              when m1.record_type='DEPOSIT' then m1.bank_code 
              when m1.record_type='WITHDRAW' then ifnull(m1.pre_id,'manual')
            end,
            case when bank_id is null then '' else '_PLURAL' end
            ) bank_code  
            ,m1.asset_code,m1.amount
            from novadax.nova_legal_record m1 
            join novadax.nova_customer_ext m2 on m1.customer_id=m2.customer_id
            where m2.source != 'TEST' and m1.record_note != 'OTC' and m1.record_type in ('DEPOSIT','WITHDRAW') 
            and m1.asset_code='BRL'
        )a 
        join (select @rn :=0,@mainid := '',@maintype := '') b
        order by 2,5,3
    )m1 
    where created_at>=subdate(curdate(),if(date_format(curdate(),'%%w')=0,7,date_format(curdate(),'%%w'))-1) - interval '49' day and created_at<current_date
    group by 1,2,3,4,5,6,7
    union 
    select 
        'month' as type
        , date_format(created_at,'%%Y-%%m') as period
        , record_type
        , asset_code
        , if(rnk=1,'first','not_first') as "rnk"
        , bank_name
        , bank_code
        , count(DISTINCT id) as "case"
        , count(DISTINCT if(record_status='SUCCESS',id,null)) as "succ"
        , count(DISTINCT if(record_status='FAIL',id,null)) as "fail"
        , count(DISTINCT if(record_status in ('NEDD_ADUIT','PAYED'),id,null)) as "pending"
        , sum(if(record_status in ('SUCCESS','FAIL'),TIMESTAMPDIFF(hour,created_at,finished_at),0)) as deal_hour
        , max(amount) max_amt
        , sum(amount) as total_amt
        , max(if(record_status='SUCCESS',amount,null)) succ_max_amt
        , sum(if(record_status='SUCCESS',amount,0)) as succ_amt
    from
    (
        select a.*,if(@mainid = customer_id and @maintype = record_type,@rn := @rn + 1,@rn :=1) as rnk,@mainid := customer_id,@maintype := record_type
        from 
        ( -- 用户首次存款取款记录
            select DISTINCT m1.id,m1.customer_id,
            if(m1.finished_at<m1.created_at,m1.finished_at,m1.created_at) created_at,
            m1.finished_at,m1.record_type,m1.record_status,m1.bank_name,
            concat(
            case 
              when m1.record_note like '%%ewally%%' then 'ewally' 
              when m1.record_note in ('transferAsset-out','transferAsset-in') then 'internal_transfer' 
              when m1.record_type='DEPOSIT' then m1.bank_code 
              when m1.record_type='WITHDRAW' then ifnull(m1.pre_id,'manual')
            end,
            case when bank_id is null then '' else '_PLURAL' end
            ) bank_code
            ,m1.asset_code,m1.amount
            from novadax.nova_legal_record m1 
            join novadax.nova_customer_ext m2 on m1.customer_id=m2.customer_id
            where m2.source != 'TEST' and m1.record_note != 'OTC' and m1.record_type in ('DEPOSIT','WITHDRAW') 
            and m1.asset_code='BRL'
        )a 
        join (select @rn :=0,@mainid := '',@maintype := '') b
        order by 2,5,3
    )m1 
    where created_at>=date_format(curdate(),'%%Y-%%m-01') - interval '6' month 
    and created_at<date_format(curdate(),'%%Y-%%m-01')
    group by 1,2,3,4,5,6,7
    order by 1,2,3
    """
    return sql




# 首存首取成功率
df_deposit = data_deposit1()
# print(df_deposit)



# 月度目标与实际

In [8]:
@data_util.db_source(db='novadaxbi')
def data_p_and_l(update=False):
    sql = """    
    select distinct period,type,attr,value
    from bacc_nova.ba_stat_goal_actual 
    where kyc_area !='EUROPE' and period<date_format(current_date,'%%Y-%%m-01')
    """
    return sql


result = data_p_and_l()
result = result.pivot_table(index = ['type','attr'],columns='period').stack(0).reset_index().fillna(0)
monthly_result = result.drop(['type','level_2'],axis=1)


# 输出附件

In [9]:
writer = pd.ExcelWriter('./nova_weekly_meeting_{date}.xlsx'.format(date=today.strftime('%Y%m%d')), engine='openpyxl')

def output_exl(dfs,sheet_name,k):
    i = 0
    for df in dfs:
        df.to_excel(excel_writer=writer, sheet_name=sheet_name,startrow=i,index=False,freeze_panes=(0,1))
        row = len(df.index)
        i += row + 4 
    ws = writer.sheets[sheet_name]
    for j in range(len(dfs[k].columns)):
        if j == 0:
            ws.column_dimensions[ascii_uppercase[j]].width = 30
        else:
            ws.column_dimensions[ascii_uppercase[j]].width = 20

output_exl(dfs = [df_transaction_week,df_asset_week,df_first_trade_week,df_order_otc_week],sheet_name = 'overview_week',k=1)
output_exl(dfs = [df_transaction_month,df_asset_month,df_first_trade_month,df_order_otc_month],sheet_name = 'overview_month',k=1)
output_exl(dfs = [df_trans_type],sheet_name = 'transaction_type',k=0)
output_exl(dfs = [df_symbol],sheet_name = 'transaction_symbol',k=0)
output_exl(dfs = [df_deposit],sheet_name = '存款时效',k=0)
# output_exl(dfs = [df_order_device],sheet_name = '下单时终端',k=0)
output_exl(dfs = [df_conv_detail],sheet_name = '分渠道获客转化',k=0)
output_exl(dfs = [df_kyc,df_kyc_device,df_kyc_market],sheet_name = 'kyc',k=0)
output_exl(dfs = [monthly_result],sheet_name = 'monthly_p_and_l',k=0)

writer.save()
print('check file: ./nova_weekly_meeting_{date}.xlsx'.format(date=today.strftime('%Y%m%d')))

check file: ./nova_weekly_meeting_20210823.xlsx
